# Transformacja danych energetycznych

Notebook wczytuje **surowe** pliki CSV z `datasets/raw/` (wynik `_etl.ipynb`),
wykonuje inżynierię danych (mapowanie kolumn, ujednolicenie czasu, uzupełnianie braków)
i zapisuje **gotowe** zbiory do `datasets/`.

Kolejność pipeline:
1. `_etl.ipynb` → `datasets/raw/energy_{KRAJ}_{OD}_{DO}.csv`
2. `_data-transformation.ipynb` → `datasets/energy_{KRAJ}_{OD}_{DO}.csv`
3. `_analiza.ipynb` — tylko wczytanie gotowych danych

In [ ]:
import os
import glob
import pandas as pd
import re

RAW_DIR = "datasets/raw"
OUTPUT_DIR = "datasets"
DEFAULT_TZ = "Europe/Warsaw"

In [ ]:
def handle_time_column(df: pd.DataFrame, tz: str = DEFAULT_TZ) -> pd.DataFrame:
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.set_index("time")
    df.index = df.index.tz_convert(tz)
    return df.sort_index()


def resample_to_hours(df: pd.DataFrame) -> pd.DataFrame:
    return df.resample("h").mean()


def get_null_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if df[c].isna().all()]
    


def handle_empty_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int, int]:
    missing_before = int(df.isna().sum().sum())
    df = df.ffill().bfill()
    missing_after = int(df.isna().sum().sum())
    return df, missing_before, missing_after


def normalize_energy_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = df.columns.tolist()
    cols_to_drop = []
    
    # Usuwamy kolumny zużycia (Consumption) i prognoz (Forecast) oraz duplikaty pandas (np. z końcówką .1)
    for col in cols:
        col_str = str(col)
        if "Consumption" in col_str or "Forecast" in col_str or col_str.endswith(".1") or col_str.endswith(".2"):
            cols_to_drop.append(col)
            
    # Dla kolumn OZE o czystych nazwach sprawdzamy czy istnieje dla nich kolumna "Actual Aggregated".
    # Jeśli istnieje i obie mają dane w tych samych wierszach (nakładają się w czasie), to czysta nazwa to prognoza i ją usuwamy.
    # Jeśli się nie nakładają, to są to komplementarne dane rzeczywiste z różnych lat i zachowujemy obie.
    clean_oze_types = ["Solar", "Wind Onshore", "Wind Offshore"]
    for oze in clean_oze_types:
        if oze in cols:
            actual_cols = [c for c in cols if oze in str(c) and "Actual Aggregated" in str(c)]
            if actual_cols:
                actual_col = actual_cols[0]
                overlap = ((df[oze].fillna(0) > 1.0) & (df[actual_col].fillna(0) > 1.0)).sum()
                if overlap > 24:
                    cols_to_drop.append(oze)
                    
    df = df.drop(columns=cols_to_drop, errors="ignore")
    
    def clean_name(col):
        if '(' in col and ',' in col:
            match = re.search(r"['\"](.*?)['\"]", col)
            if match:
                col = match.group(1)
        
        col = col.split(' - ')[0]
        col = col.strip()
        return col

    df = df.rename(columns=clean_name)
    df = df.T.groupby(level=0).sum().T
    
    return df
    
def add_renewable_field(df: pd.DataFrame) -> pd.DataFrame:
    renewable_keywords = [
        "Biomass",
        "Geothermal",
        "Hydro Run-of-river and poundage",
        "Hydro Water Reservoir",
        "Other renewable",
        "Solar",
        "Wind Offshore",
        "Wind Onshore"
    ]
    existing_columns = [col for col in renewable_keywords if col in df.columns]
    if existing_columns:
        df["renewable"] = df[existing_columns].sum(axis=1, min_count=1)
    else:
        df["renewable"] = 0
    return df


In [3]:
from IPython.core import inputtransformer2
from IPython.core import inputtransformer2
def generate_report(
    df: pd.DataFrame,
    raw_df_path: str,
    out_path: str,
    null_cols: list[str],
    missing_before: int,
    missing_after: int,
) -> None:
    size_mb = os.path.getsize(out_path) / (1024 * 1024)

    print(f"Zbiór danych: {raw_df_path} -> {out_path}")
    print(f"Kształt: {df.shape[0]} wierszy × {df.shape[1]} kolumn")
    print(f"Zakres:  {df.index.min()} → {df.index.max()}")
    print(f"Braki: {missing_before} → {missing_after}")
    print(f"Usunięto puste kolumny ({len(null_cols)}): {null_cols}")
    print(f"Zapisano ({size_mb:.2f} MB)")


def handle_dataset(df: pd.DataFrame, raw_path: str, out_path: str) -> pd.DataFrame:
    df = handle_time_column(df)
    df = resample_to_hours(df)
    df = normalize_energy_columns(df)
    null_cols = get_null_columns(df)
    df, missing_before, missing_after = handle_empty_rows(df)
    df = add_renewable_field(df)
    df.index.name = "time"

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    df.to_csv(out_path)
    generate_report(df, raw_path, out_path, null_cols, missing_before, missing_after)

    return df

## Przetwarzanie wszystkich plików surowych

In [4]:
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "energy_*.csv")))

for raw_path in raw_files:
    basename = os.path.basename(raw_path)
    out_path = os.path.join(OUTPUT_DIR, basename)

    df = pd.read_csv(raw_path)
    
    print(f"Wejście:  {raw_path}")
    handle_dataset(df, raw_path, out_path)
    print(f"Wyjście:  {out_path}")

Wejście:  datasets/raw/energy_BG_2020-01-01_2025-12-31.csv
Zbiór danych: datasets/raw/energy_BG_2020-01-01_2025-12-31.csv -> datasets/energy_BG_2020-01-01_2025-12-31.csv
Kształt: 52609 wierszy × 17 kolumn
Zakres:  2019-12-31 23:00:00+01:00 → 2025-12-31 23:00:00+01:00
Braki: 0 → 0
Usunięto puste kolumny (0): []
Zapisano (6.61 MB)
Wyjście:  datasets/energy_BG_2020-01-01_2025-12-31.csv
Wejście:  datasets/raw/energy_DE_2020-01-01_2025-12-31.csv
Zbiór danych: datasets/raw/energy_DE_2020-01-01_2025-12-31.csv -> datasets/energy_DE_2020-01-01_2025-12-31.csv
Kształt: 52608 wierszy × 20 kolumn
Zakres:  2020-01-01 00:00:00+01:00 → 2025-12-31 23:00:00+01:00
Braki: 0 → 0
Usunięto puste kolumny (0): []
Zapisano (11.64 MB)
Wyjście:  datasets/energy_DE_2020-01-01_2025-12-31.csv
Wejście:  datasets/raw/energy_ES_2020-01-01_2025-12-31.csv
Zbiór danych: datasets/raw/energy_ES_2020-01-01_2025-12-31.csv -> datasets/energy_ES_2020-01-01_2025-12-31.csv
Kształt: 52609 wierszy × 25 kolumn
Zakres:  2020-01-01 00